In [ ]:
def arma_covariance(M: int, ar: float, ma: float):
    C = np.zeros((M, M))
    for k in range(M):
        for h in range(M):
            if k == h:
                C[k, h] = 1 + (ar + ma) ** 2 / (1 - ar**2)
            elif abs(k - h) == 1:
                C[k, h] = ar + ma + (ar + ma) ** 2 * ar / (1 - ar**2)
            else:
                C[k, h] = ar ** (abs(k - h) - 1) * (
                    ar + ma + (ar + ma) ** 2 * ar / (1 - ar**2)
                )
    return C


def generate_data(N: int, M: int, time_ar: float, time_ma: float):
    burn = 100
    real = arma_generate_sample(
        [1, -time_ar], [1, time_ma], (N + burn, M), scale=1 / np.sqrt(2)
    )
    imag = arma_generate_sample(
        [1, -time_ar], [1, time_ma], (N + burn, M), scale=1 / np.sqrt(2)
    )
    y = real + 1j * imag

    return y[burn:] 



def run(N, M, time_ar, time_ma, space_ar, space_ma):
    space_cov = arma_covariance(M, space_ar, space_ma)
    space_cov_sqrt = sqrtm(space_cov)
    y = generate_data(N, M, time_ar, time_ma)
    y = y @ space_cov_sqrt
    gpy_result = GPY(y, fs, is_complex_gaussian=True)
    return gpy_result